# 05 · Final model: train once on all boreholes and freeze it

The model chosen in notebook 04 (model **and** feature set) is tuned again on **all** boreholes, with the same procedure as the inner CV (groups = the 5 blockCV folds), and then trained on all cells. With VA-HDW, the reference boreholes are now all boreholes.

A **comparison model** is trained the same way: the best candidate (same rule) on the **other** feature set. Notebook 07 scores both on the new boreholes, so the external test also shows whether VA-HDW helps.

Both files are then **frozen**: their SHA-256 fingerprints are stored, and notebooks 06-08 refuse to run if a file changes. Running this notebook again is refused once a final model exists (set `REFIT = True` only with a reason you report in the thesis).

**Run time:** see the README (GBARNN, GWRF and GWXGB are the slowest).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib

REFIT = False
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
sel = json.loads((H.out_dir(S, "04_results") / "selection.json").read_text())
summ = pd.read_csv(H.out_dir(S, "04_results") / "scores_mean_sd.csv")
OUT, PRIV = H.out_dir(S, "05_final_model"), H.out_dir(S, "05_final_model", private=True)
record = OUT / "final_model.json"
if record.exists() and not REFIT:
    r = json.loads(record.read_text())
    raise RuntimeError(f"A final model is already frozen ({r['model']} | {r['features']}). "
                       "Set REFIT = True only with a documented reason.")
other = "XYZ" if sel["features"] == "XYZ_VAHDW" else "XYZ_VAHDW"
cmp_sel = H.select_model(summ[summ.features == other], "all")
CHOSEN = [(sel["selected"], sel["features"], "final")]
if cmp_sel["selected"]:
    CHOSEN.append((cmp_sel["selected"], other, "comparison"))
print(f"Block size {block_m:.1f} m | {cells.BH_Code.nunique()} boreholes, {len(cells)} cells")
for name, f, role in CHOSEN:
    print(f"{role:<11} {name} | {f}")

In [ ]:
groups = cells.BH_Code.map(folds).values
frozen = {}
for name, features, role in CHOSEN:
    seed = H.seed_of(S, "final", name, features)
    best = H.tune(name, features, cells, groups, seed, S)
    model = H.make_model(name, features, best["params"], seed, S).fit(cells, H.fill_weights(cells, best["params"]))
    if hasattr(model.base, "local_"):       # GW models: train every local model now, so the frozen file is complete
        model.p_clay(cells)
    path = PRIV / f"{role}_{name}_{features}.joblib"
    joblib.dump(model, path, compress=3)
    frozen[role] = {"model": name, "features": features, "params": best["params"], "threshold": best["threshold"],
                    "inner_macro_f1": best["inner_macro_f1"], "file": path.name, "sha256": H.sha256(path)}
    best["trials"].to_csv(OUT / f"final_trials_{role}_{name}_{features}.csv", index=False)
    print(f"{role}: {name} | {features}: threshold {best['threshold']:.2f}, inner Macro-F1 {best['inner_macro_f1']:.3f}, "
          f"SHA-256 {frozen[role]['sha256'][:16]}")

In [ ]:
rec = {**frozen["final"], "comparison": frozen.get("comparison"), "block_m": block_m, "area": S["area"]["area"],
       "train_boreholes": int(cells.BH_Code.nunique()), "train_cells": len(cells),
       "train_filled_cells": int((cells.measured == 0).sum()),
       "top_elevation_range_m": [float(cells.Top_Elevation_m.min()), float(cells.Top_Elevation_m.max())],
       "frozen_on": time.strftime("%Y-%m-%d %H:%M"), "quick": S["run"]["quick"]}
record.write_text(json.dumps(rec, indent=1, default=str))
print(json.dumps(rec, indent=1, default=str))
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 06.")